# 00 — Pull raw data

**Inputs:** none (downloads from uscis.gov; DOL workbooks acquired manually)
**Function:** fetch the USCIS H-1B Employer Data Hub files, verify what arrived, and document the DOL LCA acquisition
**Outputs:** `data/uscis_2017.csv` … `data/uscis_2023.csv`

The project joins two datasets that the U.S. government publishes separately and never
links: USCIS adjudication outcomes (who was denied) and DOL Labor Condition Application
filings (what the job paid relative to the prevailing wage). This notebook acquires both.

In [1]:
import sys
from pathlib import Path

# Locate the repository root by walking up until code/src is found, then put the
# code directory on the path. No absolute paths, so this runs from any checkout.
_here = Path.cwd().resolve()
_root = next(p for p in (_here, *_here.parents) if (p / "code" / "src").is_dir())
sys.path.insert(0, str(_root / "code"))

import pandas as pd
import requests

from src import DATA, LCA_RAW, USCIS_YEARS, describe_frame


def expected_lca_files():
    """The fifteen workbook names the pipeline expects, in order."""
    names = [f"LCA_FY{fy}.xlsx" for fy in (2017, 2018, 2019)]
    for fy in (2020, 2021, 2022):
        names += [f"LCA_FY{fy}_Q{q}.xlsx" for q in (1, 2, 3, 4)]
    return names


## 1. USCIS H-1B Employer Data Hub

One CSV per fiscal year, FY2017 through FY2023. USCIS builds these from its adjudication
systems and publishes them as a transparency product. Each row is an employer at a
worksite in a fiscal year, with four counts: initial approvals, initial denials,
continuing approvals, continuing denials.

FY2023 is a partial year. It is pulled for completeness but excluded from every model.

In [2]:
USCIS_URL = "https://www.uscis.gov/sites/default/files/document/data/Employer_Information_{fy}.csv"


def pull_uscis(fy, overwrite=False):
    """Download one fiscal year of the USCIS Employer Data Hub.

    Returns the destination path. Existing files are kept unless overwrite=True,
    so re-running the notebook is cheap and does not re-hit uscis.gov.
    """
    dest = DATA / f"uscis_{fy}.csv"
    if dest.exists() and not overwrite:
        print(f"FY{fy}: already present ({dest.stat().st_size / 1e6:.1f} MB), skipping")
        return dest
    resp = requests.get(USCIS_URL.format(fy=fy), timeout=120)
    resp.raise_for_status()
    dest.write_bytes(resp.content)
    print(f"FY{fy}: downloaded {len(resp.content) / 1e6:.1f} MB -> {dest.name}")
    return dest


for fy in USCIS_YEARS:
    pull_uscis(fy)

FY2017: already present (4.3 MB), skipping
FY2018: already present (4.8 MB), skipping
FY2019: already present (5.1 MB), skipping
FY2020: already present (3.6 MB), skipping
FY2021: already present (4.0 MB), skipping
FY2022: already present (4.0 MB), skipping
FY2023: already present (2.2 MB), skipping


### Verify what arrived

Diagnostic before anything downstream touches these files: row counts per year, and the
column headers, which are **not** stable across years. USCIS renamed "Initial Approvals"
to "Initial Approval" between FY2019 and FY2020, and notebook `02_merge` reconciles that.

In [3]:
total = 0
for fy in USCIS_YEARS:
    d = pd.read_csv(DATA / f"uscis_{fy}.csv", dtype=str, nrows=None)
    total += len(d)
    print(f"FY{fy}: {len(d):>7,} rows   {d.shape[1]} columns")
print(f"\nTotal raw rows across all years: {total:,}")

# Show the header drift explicitly rather than assuming it.
h2019 = set(pd.read_csv(DATA / "uscis_2019.csv", dtype=str, nrows=0).columns)
h2020 = set(pd.read_csv(DATA / "uscis_2020.csv", dtype=str, nrows=0).columns)
print("\nIn FY2019 but not FY2020:", sorted(h2019 - h2020))
print("In FY2020 but not FY2019:", sorted(h2020 - h2019))

FY2017:  49,786 rows   11 columns
FY2018:  55,666 rows   11 columns
FY2019:  59,441 rows   11 columns
FY2020:  55,239 rows   11 columns
FY2021:  60,806 rows   11 columns


FY2022:  59,983 rows   11 columns
FY2023:  33,332 rows   11 columns

Total raw rows across all years: 374,253

In FY2019 but not FY2020: ['Continuing Approvals', 'Continuing Denials', 'Initial Approvals', 'Initial Denials']
In FY2020 but not FY2019: ['Continuing Approval', 'Continuing Denial', 'Initial Approval', 'Initial Denial']


In [4]:
sample = pd.read_csv(DATA / "uscis_2019.csv", dtype=str)
describe_frame(sample, "USCIS FY2019 raw", keys=["Employer", "State", "NAICS"])
sample.head(3)

--- USCIS FY2019 raw ---
    rows: 59,441   columns: 11
    distinct Employer: 51,886
    distinct State: 55
    distinct NAICS: 25
    columns with missing values: 4 (worst: Tax ID at 0.4%)


,Fiscal Year,Employer,Initial Approvals,Initial Denials,Continuing Approvals,Continuing Denials,NAICS,Tax ID,State,City,ZIP
0,2019,SOUTHERN CARPET HARDWOOD & TILE IN,1,0,0,0,23,NaN,AL,BIRMINGHAM,35209
1,2019,UAB HEALTH SYSTEM,0,0,0,1,56,NaN,AL,BIRMINGHAM,35233
2,2019,BIRMINGHAM VA MEDICAL CENTER,0,0,1,0,62,NaN,AL,BIRMINGHAM,35233


## 2. DOL LCA disclosure data — manual acquisition

Before an employer can file an H-1B petition it must file a Labor Condition Application
with the Department of Labor stating the offered wage and the prevailing wage for that
occupation and location. DOL publishes every one of these.

**These files cannot be downloaded by script.** `www.dol.gov` returns HTTP 403 to
programmatic requests, and the old OFLC Data Center domain no longer resolves. The
fifteen workbooks were downloaded manually through a browser from the OFLC performance
data page and placed in `data/lca/`. They total ~1.8 GB and are therefore excluded from
this repository — the cell below checks for them and tells you what is missing.

Two structural facts about these files drive the extraction step in notebook `01`:

- FY2017–FY2019 are published as **one cumulative annual workbook** each.
- From FY2020 DOL switched to **one workbook per quarter**, so those years need all four
  quarters concatenated. Using only the Q4 file — the first approach tried — captures
  roughly a quarter of each recent year.

In [5]:
present, missing, size = [], [], 0
for name in expected_lca_files():
    p = LCA_RAW / name
    if p.exists():
        present.append(name)
        size += p.stat().st_size
        print(f"  present  {name:24s} {p.stat().st_size / 1e6:7.1f} MB")
    else:
        missing.append(name)
        print(f"  MISSING  {name}")

print(f"\n{len(present)}/15 workbooks present, {size / 1e9:.2f} GB total")
if missing:
    print("\nDownload the missing files from the DOL OFLC performance data page:")
    print("  https://www.dol.gov/agencies/eta/foreign-labor/performance")
    print(f"and save them to {LCA_RAW} using the names listed above.")

  present  LCA_FY2017.xlsx            171.8 MB
  present  LCA_FY2018.xlsx            177.6 MB
  present  LCA_FY2019.xlsx            283.2 MB
  present  LCA_FY2020_Q1.xlsx          77.7 MB
  present  LCA_FY2020_Q2.xlsx          76.8 MB
  present  LCA_FY2020_Q3.xlsx          93.3 MB
  present  LCA_FY2020_Q4.xlsx          56.0 MB
  present  LCA_FY2021_Q1.xlsx          59.7 MB
  present  LCA_FY2021_Q2.xlsx         164.9 MB
  present  LCA_FY2021_Q3.xlsx         289.7 MB
  present  LCA_FY2021_Q4.xlsx          96.6 MB
  present  LCA_FY2022_Q1.xlsx          56.2 MB
  present  LCA_FY2022_Q2.xlsx          71.1 MB
  present  LCA_FY2022_Q3.xlsx         109.5 MB
  present  LCA_FY2022_Q4.xlsx          56.1 MB

15/15 workbooks present, 1.84 GB total


**Next:** `01_extract_lca.ipynb` streams these workbooks down to the ~20 columns the
analysis needs.